![FLIP Banner](../../Assets/images/flip-banner.png)

# FLIP: Agentic AI in Practice
**(Module 01: Foundations)**

---

- Materials in this module have been developed to support practical learning in generative AI and agentic AI systems.
- You are free to use, modify and distribute this package for teaching, learning and research purposes.
- If you find any issue or bug in this document, please submit an issue at [tulip-lab/agentic-ai](https://github.com/tulip-lab/agentic-ai/issues).

Prepared by :tulip: **[TULIP Lab](https://www.tulip.academy), Australia**

---

## Session 1C: LLMs, Function Calling, APIs and Agent Ecosystems

<div align="center">

<table>
<thead>
<tr>
<th><strong>Item</strong></th>
<th><strong>Description</strong></th>
</tr>
</thead>
<tbody>
<tr><td align="left">Estimated time</td><td>2 hours</td></tr>
<tr><td align="left">Environment</td><td>Google Colab or local Jupyter</td></tr>
<tr><td align="left">Main output</td><td>A safe mock function-calling router that validates tool names and arguments</td></tr>
</tbody>
</table>

</div>

---

**Table of Contents**

1. [Overview and Learning Goals](#m01c-overview)
2. [Setup and Background](#m01c-setup)
3. [Core Concepts](#m01c-core-concepts)
4. [Guided Implementation](#m01c-guided-implementation)
5. [Testing and Analysis](#m01c-testing)
6. [Student Tasks](#m01c-student-tasks)
7. [Submission and Reflection](#m01c-submission)

---

<a id="m01c-overview"></a>

### 1. Overview and Learning Goals

This session connects the safety patterns from `M01A` and the conceptual distinctions from `M01B` to the practical structure of LLM APIs and function calling. In later sessions, you will use Flowise, LangChain, LangGraph, CrewAI, AutoGen and other agent-building tools. Those frameworks may look different at the interface level, but they share a common pattern: a model receives a task, a workflow decides whether a tool should be called, the tool input is validated, and the result is returned in a structured form.

In this session, we do not call a real LLM API. Instead, we simulate the kind of tool-call message that an LLM API might return. This is deliberate. A mock implementation lets you inspect the control flow without needing API keys, network access or provider-specific syntax. Once you understand this structure, later practicals can replace the mock response with a real model response.

The core question for this lab is not “which model provider should we use?” The core question is: **what must happen between a model suggesting a tool call and the system actually executing that tool?** A safe agentic system should not blindly trust a model-generated function call. It should check that the tool is allowed, validate the arguments, reject invalid requests and return a structured response.

By the end of this lab, you should be able to explain the structure of an LLM API request, distinguish prompting from function calling, explain why tool schemas and argument validation are needed, compare common agent frameworks at a conceptual level, build a mock function-calling router, and test normal, edge and failure behaviours.

<a id="m01c-setup"></a>

### 2. Setup and Background

This notebook uses only the Python standard library. It does not require an LLM API key. This keeps the session safe and reproducible, and it allows you to focus on the design pattern rather than provider-specific SDK syntax.

Later sessions will use real external services and frameworks. For example, [M03D-Flowise-AgentFlow-Tools](../../Flowise/M03-VisualWorkflows/M03D-Flowise-AgentFlow-Tools.md) will use tool nodes in Flowise, [M04B-LangChain-ToolAgents](../M04-LangChain/M04B-LangChain-ToolAgents.ipynb) will build tool-using agents in Python, and [M05C-LangGraph-StatefulWorkflows](../M05-KnowledgeState/M05C-LangGraph-StatefulWorkflows.ipynb) will formalise multi-step stateful execution.

The examples in this notebook are self-contained. When data or documents are needed in later practicals, use public unit materials first, then public datasets from [tulip-lab/open-data](https://github.com/tulip-lab/open-data).

<div align="center">

<table>
<thead>
<tr><th><strong>Component</strong></th><th><strong>Role in function calling</strong></th><th><strong>Safety concern</strong></th></tr>
</thead>
<tbody>
<tr><td align="left">User request</td><td>Describes what the user wants.</td><td>May be ambiguous, malicious or unsupported.</td></tr>
<tr><td align="left">LLM response</td><td>May suggest a tool name and arguments.</td><td>May be wrong or unsafe.</td></tr>
<tr><td align="left">Tool registry</td><td>Defines which tools are allowed.</td><td>Unknown tools must be rejected.</td></tr>
<tr><td align="left">Schema or validator</td><td>Checks tool arguments before execution.</td><td>Missing, wrong-type or invalid values must be rejected.</td></tr>
<tr><td align="left">Tool execution</td><td>Runs the approved function.</td><td>Should produce structured output.</td></tr>
</tbody>
</table>

</div>

In [ ]:
# This session uses only the Python standard library.
# Later sessions will introduce real LLM APIs and agent frameworks.

import math
from typing import Any, Callable, Dict

print("Setup complete.")

<a id="m01c-core-concepts"></a>

### 3. Core Concepts

An LLM API request usually contains a model name, a list of messages, optional generation parameters and sometimes a list of available tools. The exact syntax differs across providers, but the conceptual structure is similar.

A minimal chat-style request can be understood as:

```python
{
    "model": "some-llm",
    "messages": [
        {"role": "system", "content": "You are a careful assistant."},
        {"role": "user", "content": "Explain what RAG means."}
    ],
    "temperature": 0.2
}
```

This is ordinary prompting. The model receives text and returns text. Function calling adds an additional possibility: instead of directly answering, the model may produce a structured suggestion to call a tool.

A simplified function-calling response may look like:

```python
{
    "tool_name": "circle_area",
    "arguments": {
        "radius": 3
    }
}
```

This response is not the same as executing the function. It is only a proposed action. A safe system must decide whether to accept or reject the proposed action.

```mermaid
flowchart LR
    A[User request] --> B[LLM or mock model]
    B --> C[Proposed tool call]
    C --> D{Tool allowed?}
    D -->|No| E[Reject safely]
    D -->|Yes| F{Arguments valid?}
    F -->|No| G[Reject with error]
    F -->|Yes| H[Execute tool]
    H --> I[Return structured result]
```

The tool registry is the boundary between model output and system action. A model can suggest any text, but the runtime should only execute tools that the developer has explicitly approved. This is why agentic AI is not just “LLM plus tools”. It is controlled execution around a model.

Common agent frameworks expose this pattern in different ways.

<div align="center">

<table>
<thead>
<tr><th><strong>Framework</strong></th><th><strong>Main role in this unit</strong></th><th><strong>Conceptual emphasis</strong></th></tr>
</thead>
<tbody>
<tr><td align="left">Flowise</td><td>Visual construction of chatbots, RAG and agent flows.</td><td>Low-code workflow composition.</td></tr>
<tr><td align="left">LangChain</td><td>Python programming for prompts, models, tools, retrieval and agents.</td><td>Composable LLM application components.</td></tr>
<tr><td align="left">LangGraph</td><td>Stateful multi-step agentic workflows.</td><td>Explicit state, graph control flow and durable execution.</td></tr>
<tr><td align="left">CrewAI</td><td>Role-based multi-agent collaboration.</td><td>Agents, roles, tasks and coordination.</td></tr>
<tr><td align="left">AutoGen</td><td>Conversational multi-agent systems.</td><td>Message-based agent interaction and coordination.</td></tr>
</tbody>
</table>

</div>

This session does not require you to master these frameworks. The goal is to understand the shared design pattern so that later framework-specific practicals make sense.

In [ ]:
# A safe structured-output helper.
# We reuse the same interface style introduced in earlier sessions.

def ok_result(value: Any) -> Dict[str, Any]:
    return {
        "ok": True,
        "error": None,
        "result": value
    }


def error_result(message: str) -> Dict[str, Any]:
    return {
        "ok": False,
        "error": message,
        "result": None
    }


print(ok_result({"example": 1}))
print(error_result("Example error."))

The output shows two structured dictionaries. A successful operation uses `ok=True`, `error=None` and stores the value in `result`. A failed operation uses `ok=False`, places a clear message in `error`, and sets `result=None`.

This structure is simple, but it is central to agentic workflow design. It lets the next step in a workflow check whether the previous step succeeded without trying to interpret free-form natural language.

<a id="m01c-guided-implementation"></a>

### 4. Guided Implementation

We now build a small mock function-calling system. The system will include approved tools, a tool registry, argument validators, a router that receives a proposed tool call, and tests for normal, edge and failure behaviours.

The first approved tool is `circle_area`, which connects directly to the M01A pattern. The mock model will suggest tool calls, but the router will decide whether execution is allowed.

In [ ]:
def circle_area(radius: Any) -> Dict[str, Any]:
    """Compute the area of a circle with input validation."""
    if not isinstance(radius, (int, float)):
        return error_result("Radius must be numeric.")
    if radius < 0:
        return error_result("Radius must not be negative.")
    return ok_result(math.pi * radius * radius)


# The tool registry is the allow-list.
# Only functions registered here can be called by the router.
TOOL_REGISTRY: Dict[str, Callable[..., Dict[str, Any]]] = {
    "circle_area": circle_area
}

print("Registered tools:", list(TOOL_REGISTRY.keys()))

The registry is intentionally explicit. A model may suggest many possible tool names, but the runtime will only execute tools that appear in `TOOL_REGISTRY`. This protects the system from executing arbitrary or unsupported actions.

In [ ]:
def validate_circle_area_arguments(arguments: Any) -> Dict[str, Any]:
    """Validate arguments for circle_area."""
    if not isinstance(arguments, dict):
        return error_result("Tool arguments must be provided as a dictionary.")

    if "radius" not in arguments:
        return error_result("Missing required argument: radius.")

    radius = arguments["radius"]

    if not isinstance(radius, (int, float)):
        return error_result("radius must be numeric.")

    if radius < 0:
        return error_result("radius must not be negative.")

    return ok_result({"radius": radius})


VALIDATORS: Dict[str, Callable[[Any], Dict[str, Any]]] = {
    "circle_area": validate_circle_area_arguments
}

print("Registered validators:", list(VALIDATORS.keys()))

The validator checks the proposed arguments before the tool is called. Notice that validation happens outside the tool even though the tool also validates its input. This duplication is acceptable in a teaching example because it shows two safety layers.

In production systems, validation may occur at several levels: API schema, router validation, tool-level validation and post-execution checks. Redundant safety checks can be useful when a tool has side effects or when model-generated arguments may be unreliable.

In [ ]:
def route_tool_call(tool_call: Dict[str, Any]) -> Dict[str, Any]:
    """Route a proposed tool call to an approved tool after validation.

    Expected input format:
    {
        "tool_name": "circle_area",
        "arguments": {"radius": 3}
    }
    """
    if not isinstance(tool_call, dict):
        return error_result("Tool call must be a dictionary.")

    tool_name = tool_call.get("tool_name")
    arguments = tool_call.get("arguments")

    if not isinstance(tool_name, str) or not tool_name:
        return error_result("Tool call must include a non-empty string tool_name.")

    if tool_name not in TOOL_REGISTRY:
        return error_result(f"Unknown or unapproved tool: {tool_name}")

    validator = VALIDATORS.get(tool_name)
    if validator is None:
        return error_result(f"No validator registered for tool: {tool_name}")

    validation = validator(arguments)
    if not validation["ok"]:
        return validation

    validated_arguments = validation["result"]
    tool_function = TOOL_REGISTRY[tool_name]

    # The validated argument dictionary is unpacked into keyword arguments.
    # This is safer than passing the raw model-generated dictionary directly.
    return tool_function(**validated_arguments)


mock_tool_call = {
    "tool_name": "circle_area",
    "arguments": {"radius": 3}
}

result = route_tool_call(mock_tool_call)
result

The output should be a successful structured result. The numeric result is approximately `28.2743`, because the area is `π × 3²`. More importantly, the result is wrapped in the same structured format used throughout this module.

The router performed four checks before execution: the tool call was a dictionary; the tool name was present and non-empty; the tool name appeared in the approved registry; and the arguments passed validation. Only after these checks did the system call `circle_area`.

In [ ]:
def mock_llm_response(user_request: str) -> Dict[str, Any]:
    """A tiny mock model that maps simple user requests to tool-call proposals.

    This is not an LLM. It is a deterministic teaching substitute.
    The purpose is to simulate the shape of an LLM function-calling response.
    """
    text = user_request.lower()

    if "circle" in text and "radius 3" in text:
        return {
            "tool_name": "circle_area",
            "arguments": {"radius": 3}
        }

    if "circle" in text and "radius 0" in text:
        return {
            "tool_name": "circle_area",
            "arguments": {"radius": 0}
        }

    if "circle" in text and "radius -1" in text:
        return {
            "tool_name": "circle_area",
            "arguments": {"radius": -1}
        }

    return {
        "tool_name": "unknown_tool",
        "arguments": {}
    }


user_request = "Calculate the area of a circle with radius 3."
proposed_call = mock_llm_response(user_request)
print("Proposed call:", proposed_call)
print("Router result:", route_tool_call(proposed_call))

This cell simulates the distinction between model output and system execution. The mock model proposes a tool call; the router decides whether the proposal is safe and executable. In a later LangChain or LangGraph practical, the mock response can be replaced with a real model response, but the validation boundary should remain.

<a id="m01c-testing"></a>

### 5. Testing and Analysis

The router should be tested with normal, edge and failure cases. The purpose is to check both functional correctness and safe rejection behaviour.

<div align="center">

<table>
<thead>
<tr><th><strong>Case</strong></th><th><strong>Example</strong></th><th><strong>Expected behaviour</strong></th></tr>
</thead>
<tbody>
<tr><td align="left">Normal</td><td><code>{"tool_name": "circle_area", "arguments": {"radius": 3}}</code></td><td>Return a successful area result.</td></tr>
<tr><td align="left">Edge</td><td><code>{"tool_name": "circle_area", "arguments": {"radius": 0}}</code></td><td>Return area <code>0</code>.</td></tr>
<tr><td align="left">Failure</td><td><code>{"tool_name": "circle_area", "arguments": {"radius": -1}}</code></td><td>Reject negative radius.</td></tr>
<tr><td align="left">Failure</td><td><code>{"tool_name": "circle_area", "arguments": {}}</code></td><td>Reject missing argument.</td></tr>
<tr><td align="left">Failure</td><td><code>{"tool_name": "delete_files", "arguments": {}}</code></td><td>Reject unknown or unapproved tool.</td></tr>
</tbody>
</table>

</div>

In [ ]:
# Normal case: approved tool and valid argument.
normal = route_tool_call({
    "tool_name": "circle_area",
    "arguments": {"radius": 3}
})
assert normal["ok"] is True
assert round(normal["result"], 4) == round(math.pi * 9, 4)

# Edge case: radius 0 is valid.
edge = route_tool_call({
    "tool_name": "circle_area",
    "arguments": {"radius": 0}
})
assert edge["ok"] is True
assert edge["result"] == 0

# Failure case: negative radius.
negative = route_tool_call({
    "tool_name": "circle_area",
    "arguments": {"radius": -1}
})
assert negative["ok"] is False
assert negative["result"] is None

# Failure case: missing required argument.
missing = route_tool_call({
    "tool_name": "circle_area",
    "arguments": {}
})
assert missing["ok"] is False
assert missing["result"] is None

# Failure case: unknown or unapproved tool.
unknown = route_tool_call({
    "tool_name": "delete_files",
    "arguments": {}
})
assert unknown["ok"] is False
assert unknown["result"] is None

print("Router tests passed.")

If the cell prints `Router tests passed.`, the router has satisfied the expected behaviours listed above. This does not mean the system is production-ready. It means the controlled behaviours in this notebook are working.

The most important result is not the circle area. The important result is that invalid or unapproved tool calls are rejected before execution. This is the same principle that will be needed when the tools become more powerful, such as search, file writing, code execution or API calls.

<a id="m01c-student-tasks"></a>

### 6. Student Tasks

Extend the mock function-calling system by adding a new approved tool:

```text
rectangle_area(width, height)
```

The new tool should calculate the area of a rectangle. It must follow the same design pattern as `circle_area`: validate inputs, reject invalid values and return structured output.

<div align="center">

<table>
<thead>
<tr><th><strong>Task</strong></th><th><strong>What you need to do</strong></th><th><strong>Why it matters</strong></th><th><strong>Expected evidence</strong></th></tr>
</thead>
<tbody>
<tr><td align="left">Task 1</td><td>Implement <code>rectangle_area(width, height)</code>.</td><td>Practises safe tool implementation.</td><td>The function returns <code>ok/error/result</code>.</td></tr>
<tr><td align="left">Task 2</td><td>Create a validator for rectangle arguments.</td><td>Separates model-generated input from trusted tool execution.</td><td>Missing, non-numeric and negative arguments are rejected.</td></tr>
<tr><td align="left">Task 3</td><td>Register the new tool and validator.</td><td>The router can only call approved tools.</td><td><code>rectangle_area</code> appears in both registries.</td></tr>
<tr><td align="left">Task 4</td><td>Run normal, edge and failure tests.</td><td>Shows safe behaviour before later real API use.</td><td>All required tests pass.</td></tr>
</tbody>
</table>

</div>

In [ ]:
# Student task starter.
# Implement rectangle_area and its validator.

# TODO: implement the rectangle tool.
# def rectangle_area(width: Any, height: Any) -> Dict[str, Any]:
#     ...

# TODO: implement the argument validator.
# def validate_rectangle_area_arguments(arguments: Any) -> Dict[str, Any]:
#     ...

# TODO: register the tool and validator.
# TOOL_REGISTRY["rectangle_area"] = rectangle_area
# VALIDATORS["rectangle_area"] = validate_rectangle_area_arguments

In [ ]:
# Student task tests.
# Uncomment and run after completing the implementation and registration.

# normal_rectangle = route_tool_call({
#     "tool_name": "rectangle_area",
#     "arguments": {"width": 3, "height": 4}
# })
# assert normal_rectangle["ok"] is True
# assert normal_rectangle["result"] == 12

# edge_rectangle = route_tool_call({
#     "tool_name": "rectangle_area",
#     "arguments": {"width": 0, "height": 4}
# })
# assert edge_rectangle["ok"] is True
# assert edge_rectangle["result"] == 0

# failure_negative = route_tool_call({
#     "tool_name": "rectangle_area",
#     "arguments": {"width": -1, "height": 4}
# })
# assert failure_negative["ok"] is False

# failure_missing = route_tool_call({
#     "tool_name": "rectangle_area",
#     "arguments": {"width": 3}
# })
# assert failure_missing["ok"] is False

# failure_unknown = route_tool_call({
#     "tool_name": "unknown_tool",
#     "arguments": {}
# })
# assert failure_unknown["ok"] is False

# print("Rectangle tool tests passed.")

<a id="m01c-submission"></a>

### 7. Submission and Reflection

Submit the completed notebook with the following evidence.

<div align="center">

<table>
<thead>
<tr><th><strong>Required item</strong></th><th><strong>What to submit</strong></th><th><strong>Quality check</strong></th></tr>
</thead>
<tbody>
<tr><td align="left">Completed tool</td><td><code>rectangle_area(width, height)</code>.</td><td>Returns structured output and validates domain constraints.</td></tr>
<tr><td align="left">Completed validator</td><td><code>validate_rectangle_area_arguments(arguments)</code>.</td><td>Rejects missing, non-numeric and negative arguments.</td></tr>
<tr><td align="left">Registry update</td><td>Tool and validator are added to the registries.</td><td>The router can call the new approved tool.</td></tr>
<tr><td align="left">Tests</td><td>Normal, edge and failure tests.</td><td>All tests run without unexpected errors.</td></tr>
<tr><td align="left">Reflection</td><td>150 to 250 words.</td><td>Reflection connects function calling to safe agentic AI design.</td></tr>
</tbody>
</table>

</div>

Reflection questions:

1. Why should a model-generated tool call not be executed directly?
2. What is the role of the tool registry?
3. Why is argument validation needed before tool execution?
4. What is the difference between a missing argument and an invalid argument?
5. How does this mock router prepare you for later Flowise, LangChain and LangGraph practicals?

#### Further Readings

- OpenAI function calling and tools documentation: <https://platform.openai.com/docs/guides/function-calling>
- Google Gemini function calling documentation: <https://ai.google.dev/gemini-api/docs/function-calling>
- LangChain tools documentation: <https://python.langchain.com/docs/concepts/tools/>
- LangGraph documentation: <https://langchain-ai.github.io/langgraph/>
- Flowise documentation: <https://docs.flowiseai.com>